# 2. ETF Screening and Selection

## Overview
This notebook implements the ETF screening and selection process across five asset classes:

1. **Equities** — distributing UCITS ETFs; filtered by platform availability, TER, dividends, size, and risk-adjusted returns
2. **Bonds** — distributing UCITS bond ETFs; same screening pipeline as equities with a wider Sharpe sensitivity (±0.25)
3. **Precious Metals** — physical gold/silver/platinum ETCs; accumulating allowed; screened globally by size and TER
4. **Energy** — crude oil / natural gas ETCs; accumulating allowed; keyword-filtered from commodities universe by ETF name
5. **Agriculture** — agricultural commodity ETCs; accumulating allowed; keyword-filtered from commodities universe by ETF name

Each asset class produces a ranked shortlist saved to `data/intermediate/` (CSV backup) and to the `screened_etfs` table in the SQLite database.

## Setup and Data Loading

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

import os
import json
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import requests
from scipy import stats

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_provider import DataProvider
from etf_utils.data_io import get_region_category_from_filename, get_asset_class_from_filename
from etf_utils.metrics import calculate_annualized_volatility
from etf_utils.platform_check import check_etf_availability
from etf_utils.database import save_screened_etfs

provider = DataProvider()

## Platform Availability Check
First, we'll define a function to check if ETFs are available on our chosen platform (InvestEngine).

> **Note:** Platform availability filtering applies to **all five asset classes**. For equities and bonds it runs after the screening loop; for precious metals, energy, and agriculture it is applied before the final `head(2)` selection so that unavailable ETCs are never included in the portfolio.

In [ ]:
# check_etf_availability is imported from etf_utils.platform_check
# Example usage:
# check_etf_availability("VEVE")  # returns True/False

In [ ]:
# Calculate benchmark returns and volatility for 2025
benchmark_year_start = "2025-01-01"
benchmark_year_end = "2025-12-31"

eq_benchmark_symbol     = "VEVE"   # VEVE.L on yfinance
bnd_benchmark_symbol    = "SAAA"   # SAAA.L on yfinance
pm_benchmark_symbol     = "SGLN"   # iShares Physical Gold ETC
energy_benchmark_symbol = "CRUD"   # WisdomTree WTI Crude Oil ETC
agri_benchmark_symbol   = "AIGA"   # WisdomTree Agriculture ETC

eq_prices     = provider.get_historical_prices(eq_benchmark_symbol, benchmark_year_start, benchmark_year_end)
bnd_prices    = provider.get_historical_prices(bnd_benchmark_symbol, benchmark_year_start, benchmark_year_end)
pm_prices     = provider.get_historical_prices(pm_benchmark_symbol, benchmark_year_start, benchmark_year_end)
energy_prices = provider.get_historical_prices(energy_benchmark_symbol, benchmark_year_start, benchmark_year_end)
agri_prices   = provider.get_historical_prices(agri_benchmark_symbol, benchmark_year_start, benchmark_year_end)

def _period_metrics(prices, start, end):
    yr = prices.loc[start:end]["close"]
    ret  = round((yr.iloc[-1] - yr.iloc[0]) / yr.iloc[0] * 100, 2)
    vol  = round(calculate_annualized_volatility(yr) * 100, 2)
    sr   = round(ret / vol, 2)
    return ret, vol, sr

eq_benchmark_return,     eq_benchmark_volatility,     eq_sharpe_ratio     = _period_metrics(eq_prices,     benchmark_year_start, benchmark_year_end)
bnd_benchmark_return,    bnd_benchmark_volatility,    bond_sharpe_ratio   = _period_metrics(bnd_prices,    benchmark_year_start, benchmark_year_end)
pm_benchmark_return,     pm_benchmark_volatility,     pm_sharpe_ratio     = _period_metrics(pm_prices,     benchmark_year_start, benchmark_year_end)
energy_benchmark_return, energy_benchmark_volatility, energy_sharpe_ratio = _period_metrics(energy_prices, benchmark_year_start, benchmark_year_end)
agri_benchmark_return,   agri_benchmark_volatility,   agri_sharpe_ratio   = _period_metrics(agri_prices,   benchmark_year_start, benchmark_year_end)

print(f"2025 VEVE  return: {eq_benchmark_return}%,  vol: {eq_benchmark_volatility}%,  Sharpe: {eq_sharpe_ratio}")
print(f"2025 SAAA  return: {bnd_benchmark_return}%, vol: {bnd_benchmark_volatility}%, Sharpe: {bond_sharpe_ratio}")
print(f"2025 SGLN  return: {pm_benchmark_return}%,  vol: {pm_benchmark_volatility}%,  Sharpe: {pm_sharpe_ratio}")
print(f"2025 CRUD  return: {energy_benchmark_return}%, vol: {energy_benchmark_volatility}%, Sharpe: {energy_sharpe_ratio}")
print(f"2025 AIGA  return: {agri_benchmark_return}%, vol: {agri_benchmark_volatility}%, Sharpe: {agri_sharpe_ratio}")

In [ ]:
benchmark_etfs = [
    # Structure: [Asset Class, Region, Country, Primary Ticker, Description]
    
    # Equity Benchmarks - Developed Markets
    ['Equity', 'Developed_AmericasandUK', 'United Kingdom', 'ISF.LON', 'iShares Core FTSE 100 UCITS ETF GBP (Dist)'],
    ['Equity', 'Developed_AmericasandUK', 'United States', 'SPY', 'SPDR S&P 500 ETF Trust'],
    ['Equity', 'Developed_AmericasandUK', 'Canada', 'ZCN.TRT', 'BMO S&P/TSX Capped Composite'],    
    ['Equity', 'Developed_EMEA', 'Germany', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'France', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Italy', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Spain', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Netherlands', 'CS51.LON', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Switzerland', 'CSWG.LON', 'Amundi Index Solutions - Amundi MSCI Switzerland'],    
    ['Equity', 'Developed_APAC', 'Japan', 'XDJP.LON', 'Xtrackers Nikkei 225 UCITS ETF 1D'],
    ['Equity', 'Developed_APAC', 'Australia', 'SAUS.LON', 'iShares MSCI Australia UCITS ETF'],    
    # Equity Benchmarks - Emerging Markets
    ['Equity', 'Emerging_Americas', 'Brazil', 'IBZL.LON', 'iShares MSCI Brazil UCITS ETF'],
    ['Equity', 'Emerging_Americas', 'Mexico', 'XMEX.LON', 'iShares MSCI Mexico Capped ETF'],    
    ['Equity', 'Emerging_APACandEMEA', 'China', 'ASHR', 'XTRACKERS HARVEST CSI 300 CHINA A-SHARES ETF'],
    ['Equity', 'Emerging_APACandEMEA', 'India', 'XNIF.LON', 'Xtrackers Nifty 50 Swap UCITS ETF 1C'],
    ['Equity', 'Emerging_APACandEMEA', 'South Korea', 'EWY', 'iShares MSCI South Korea ETF'],    
]

benchmark_df = pd.DataFrame(
    benchmark_etfs, 
    columns=['asset_class', 'region', 'country', 'benchmark_ticker', 'benchmark_description']
)

# Sort the DataFrame
benchmark_df = benchmark_df.sort_values(['asset_class', 'region', 'country'])

# Calculate 2025 returns for each benchmark ETF using DataProvider
benchmark_df['benchmark_2025_Return'] = benchmark_df['benchmark_ticker'].apply(
    lambda sym: round(provider.get_benchmark_period_return(sym, "2025-01-01", "2025-12-31") * 100, 2)
)
benchmark_df

### Fetch benchmark close prices

Retrieve the previous trading day's close price for each of the five benchmark ETFs:

| Asset Class | Benchmark | Ticker |
|---|---|---|
| Equities | Vanguard FTSE Dev World | VEVE.L |
| Bonds | SPDR Bloomberg 0–3Y US Agg | SAAA.L |
| Precious Metals | iShares Physical Gold ETC | SGLN.L |
| Energy | WisdomTree WTI Crude Oil | CRUD.L |
| Agriculture | WisdomTree Agriculture | AIGA.L |

In [ ]:
# get_latest_price is now available via provider.get_latest_price(symbol)
# Example usage:
# date_str, price = provider.get_latest_price("VEVE")  # returns (date, price) tuple

## ETF Screening Process

Screen and rank ETFs within each asset class using pre-determined weights.

### Equities & Bonds
Filtered for distributing ETFs, platform availability, minimum AUM, and maximum TER. Ranked by a weighted composite of 1-year (50%), 3-year (30%), and 5-year (20%) risk-adjusted return/risk scores, adjusted by Sharpe ratio relative to the asset-class benchmark.

### Precious Metals, Energy & Agriculture
Screened globally (no regional split). Accumulating ETCs are permitted. Filtered by minimum AUM (>100 M), maximum TER (<0.60%), not hedged, and available on InvestEngine. Energy and Agriculture ETCs are keyword-filtered from the JustETF commodities universe by ETF name. Top 2 available ETCs per class are selected based on cost efficiency and risk-adjusted return.

In [ ]:
# get_region_category_from_filename and get_asset_class_from_filename
# are imported from etf_utils.data_io

main_df_equities = pd.DataFrame()

for filepath in sorted(DATA_RAW.glob("justetf_class-equity*.csv")):
    csvl = filepath.name
    try:
        asset_class = get_asset_class_from_filename(csvl)
        region_category = get_region_category_from_filename(csvl)

        if not asset_class or region_category == 'Unknown' or asset_class != 'equity':
            print(f'Skipping {csvl}')
            continue

        print(f'Processing {asset_class} file for {region_category}: {csvl}')

        try:
            csv_df = pd.read_csv(filepath)
            if csv_df.empty:
                print(f'Empty file: {csvl}')
                continue
        except pd.errors.EmptyDataError:
            print(f'Empty file: {csvl}')
            continue

        csv_df['asset_class'] = asset_class
        csv_df['region'] = csv_df['region'].fillna('N/A')
        csv_df['country'] = csv_df['country'].fillna('N/A')

        distributing_df = csv_df.copy()
        distributing_df = distributing_df[distributing_df['dividends'] == 'Distributing']
        distributing_df = distributing_df[distributing_df['size'] > 100]
        distributing_df = distributing_df[distributing_df['hedged'] == False]

        include_tickers = ['IBZL']
        distributing_df = distributing_df[
            (distributing_df['ter'] < 0.5) | (distributing_df['ticker'].isin(include_tickers))
        ]

        # Tickers to exclude (re-evaluate with yfinance — some may now be available)
        remove_tickers = ['XUCN', 'LYXIB', 'C001', 'SW2CHA', 'XSMI', 'SJPD', 'XESD', 'C024']
        distributing_df = distributing_df[~distributing_df['ticker'].isin(remove_tickers)]

        hy_df = (distributing_df.copy()
                 .sort_values(by=['last_year_dividends'], ascending=False)
                 .drop_duplicates(subset=['region_category']))
        hy_df['intra_region_category'] = 'High Yield'

        benchmark_distributing_df = distributing_df.copy()[
            ~distributing_df['country'].isin(hy_df['country'])
        ]
        benchmark_distributing_df = pd.merge(
            benchmark_distributing_df,
            benchmark_df[['country', 'benchmark_ticker', 'benchmark_description', 'benchmark_2025_Return']],
            on='country', how='left'
        )
        benchmark_distributing_df["beta"] = (
            benchmark_distributing_df["2025"] / benchmark_distributing_df["benchmark_2025_Return"]
        )

        # Save intermediate benchmark data
        intermediate_path = DATA_INTERMEDIATE / f'benchmark_distributing_df_{csvl}'
        benchmark_distributing_df.to_csv(intermediate_path, index=False)

        benchmark_distributing_df = benchmark_distributing_df[benchmark_distributing_df['beta'] >= 1]
        beta_df = (benchmark_distributing_df.copy()
                   .sort_values(by=['ter', 'beta'], ascending=[True, False])
                   .drop_duplicates(subset=['region_category']))
        beta_df['intra_region_category'] = 'Beta'

        main_df_equities = pd.concat([main_df_equities, hy_df, beta_df], ignore_index=True)

    except Exception as e:
        print(f'Error processing {csvl}: {e}')

# Check InvestEngine availability
main_df_equities['available_on_investengine'] = (
    main_df_equities['ticker'].apply(check_etf_availability)
)

# Fetch latest close price via DataProvider (yfinance, no API key)
def _get_price(ticker):
    try:
        return provider.get_latest_price(ticker)
    except Exception as e:
        print(f"Price error for {ticker}: {e}")
        return None, None

main_df_equities[['yday_close_price_date', 'yday_close_price']] = (
    main_df_equities['ticker'].apply(_get_price).to_list()
)

main_df_equities.to_csv(DATA_INTERMEDIATE / 'summary_equities.csv', index=False)
save_screened_etfs(main_df_equities, 'equity', portfolio_year=2026)
print(f"Saved {len(main_df_equities)} equity ETFs")

In [ ]:
############# BONDS #############
main_df_bonds = pd.DataFrame()

# Manually curated bond ETF list
bonds_data = {
    'asset_class': ['bonds'] * 8,
    'region_category': [
        'Developed_AmericasandUK', 'Developed_AmericasandUK',
        'Developed_AmericasandUK', 'Developed_AmericasandUK',
        'Developed_EMEA', 'Developed_EMEA', 'Developed_EMEA',
        'Emerging_APACandEMEA',
    ],
    'intra_region_category': ['Govt', 'Corp', 'Govt', 'Corp', 'Govt', 'Corp', 'High yield', 'Corp'],
    'ticker': ['IGLT', 'SLXX', 'TRXG', 'UC81', 'PRIR', 'VECP', 'JNKE', 'EMCP'],
}
df_bonds_list = pd.DataFrame(bonds_data)
df_bonds_list = df_bonds_list[df_bonds_list['ticker'] != 'JNKE']  # not on InvestEngine

for filepath in sorted(DATA_RAW.glob("justetf_class-bonds_*.csv")):
    csvl = filepath.name
    try:
        asset_class = get_asset_class_from_filename(csvl)
        region_category = get_region_category_from_filename(csvl)
        print(f'Processing {asset_class} file for {region_category}: {csvl}')

        try:
            csv_df = pd.read_csv(filepath)
            if csv_df.empty:
                print(f'Empty file: {csvl}')
                continue
        except pd.errors.EmptyDataError:
            print(f'Empty file: {csvl}')
            continue

        csv_df['asset_class'] = asset_class
        csv_df['region'] = csv_df['region'].fillna('N/A')
        csv_df['country'] = csv_df['country'].fillna('N/A')

        for _, row in df_bonds_list.iterrows():
            if row['ticker'] in csv_df['ticker'].values:
                ticker = row['ticker']
                print(f'  Found {ticker} in {csvl}')
                csv_df_ticker = csv_df[csv_df['ticker'] == ticker].copy()
                csv_df_ticker['intra_region_category'] = row['intra_region_category']
                csv_df_ticker['region_category'] = row['region_category']
                main_df_bonds = pd.concat([main_df_bonds, csv_df_ticker], ignore_index=True)

    except Exception as e:
        print(f'Error processing {csvl}: {e}')

main_df_bonds['available_on_investengine'] = (
    main_df_bonds['ticker'].apply(check_etf_availability)
)

main_df_bonds[['yday_close_price_date', 'yday_close_price']] = (
    main_df_bonds['ticker'].apply(_get_price).to_list()
)

main_df_bonds.to_csv(DATA_INTERMEDIATE / 'summary_bonds.csv', index=False)
save_screened_etfs(main_df_bonds, 'bonds', portfolio_year=2026)
print(f"Saved {len(main_df_bonds)} bond ETFs")

In [ ]:
############# PRECIOUS METALS #############
# Relaxed filters: allow accumulating ETCs, higher TER ceiling, no distributing requirement

pm_raw_path = DATA_RAW / 'justetf_class-preciousMetals_global.csv'
main_df_pm = pd.DataFrame()

if pm_raw_path.exists():
    pm_df = pd.read_csv(pm_raw_path)
    pm_df['asset_class'] = 'preciousMetals'
    pm_df['region_category'] = 'Global'
    pm_df['intra_region_category'] = 'Global'
    pm_df['region'] = pm_df.get('region', 'Global')
    pm_df['country'] = pm_df.get('country', 'Global')

    pm_filtered = pm_df[
        (pm_df['size'] > 100) &
        (pm_df['ter'] < 0.6) &
        (pm_df['hedged'] == False)
    ].copy()

    # Rank by TER ascending, then last_year_return_per_risk descending
    pm_filtered = pm_filtered.sort_values(
        ['ter', 'last_year_return_per_risk'], ascending=[True, False]
    )

    # Filter to InvestEngine-available ETCs before selecting top 2
    pm_filtered['available_on_investengine'] = pm_filtered['ticker'].apply(check_etf_availability)
    pm_filtered = pm_filtered[pm_filtered['available_on_investengine'] == True]
    main_df_pm = pm_filtered.head(2).copy()

    main_df_pm[['yday_close_price_date', 'yday_close_price']] = (
        main_df_pm['ticker'].apply(_get_price).to_list()
    )

    main_df_pm.to_csv(DATA_INTERMEDIATE / 'summary_preciousMetals.csv', index=False)
    save_screened_etfs(main_df_pm, 'preciousMetals', portfolio_year=2026)
    print(f"Saved {len(main_df_pm)} precious metals ETCs")
    display(main_df_pm[['ticker', 'name', 'ter', 'dividends', 'size', 'last_year_return_per_risk',
                         'available_on_investengine', 'yday_close_price']])
else:
    print(f"Warning: {pm_raw_path} not found — run notebook 01 first to scrape precious metals data.")

############# ENERGY #############
# Keyword-filter commodities CSV for energy-related ETCs

cmd_raw_path = DATA_RAW / 'justetf_class-commodities_global.csv'
main_df_energy = pd.DataFrame()
main_df_agri = pd.DataFrame()

if cmd_raw_path.exists():
    cmd_df = pd.read_csv(cmd_raw_path)

    # --- Energy ---
    energy_df = cmd_df[
        cmd_df['name'].str.contains('Oil|Crude|Gas|Energy|Petroleum', case=False, regex=True) &
        ~cmd_df['name'].str.contains(
            'ex-Agriculture|Transition Metal|Industrial Metal|Copper|Nickel|Zinc|Aluminium',
            case=False, regex=True
        )
    ].copy()

    energy_filtered = energy_df[
        (energy_df['size'] > 100) &
        (energy_df['ter'] < 0.6) &
        (energy_df['hedged'] == False)
    ].sort_values(['ter', 'last_year_return_per_risk'], ascending=[True, False])

    # Filter to InvestEngine-available ETCs before selecting top 2
    energy_filtered['available_on_investengine'] = energy_filtered['ticker'].apply(check_etf_availability)
    energy_filtered = energy_filtered[energy_filtered['available_on_investengine'] == True]

    main_df_energy = energy_filtered.head(2).copy()
    main_df_energy['asset_class'] = 'energy'
    main_df_energy['region_category'] = 'Global'
    main_df_energy['intra_region_category'] = 'Global'
    main_df_energy['region'] = main_df_energy.get('region', 'Global')
    main_df_energy['country'] = main_df_energy.get('country', 'Global')
    main_df_energy[['yday_close_price_date', 'yday_close_price']] = (
        main_df_energy['ticker'].apply(_get_price).to_list()
    )

    main_df_energy.to_csv(DATA_INTERMEDIATE / 'summary_energy.csv', index=False)
    save_screened_etfs(main_df_energy, 'energy', portfolio_year=2026)
    print(f"Saved {len(main_df_energy)} energy ETCs")
    display(main_df_energy[['ticker', 'name', 'ter', 'dividends', 'size', 'last_year_return_per_risk',
                             'available_on_investengine', 'yday_close_price']])

    ############# AGRICULTURE #############

    agri_df = cmd_df[
        cmd_df['name'].str.contains(
            'Agri|Agriculture|Wheat|Grain|Corn|Soy|Soft|Coffee|Sugar|Cotton|Livestock',
            case=False, regex=True
        )
    ].copy()

    agri_filtered = agri_df[
        (agri_df['size'] > 100) &
        (agri_df['ter'] < 0.6) &
        (agri_df['hedged'] == False)
    ].sort_values(['ter', 'last_year_return_per_risk'], ascending=[True, False])

    # Filter to InvestEngine-available ETCs before selecting top 2
    agri_filtered['available_on_investengine'] = agri_filtered['ticker'].apply(check_etf_availability)
    agri_filtered = agri_filtered[agri_filtered['available_on_investengine'] == True]

    main_df_agri = agri_filtered.head(2).copy()
    main_df_agri['asset_class'] = 'agri'
    main_df_agri['region_category'] = 'Global'
    main_df_agri['intra_region_category'] = 'Global'
    main_df_agri['region'] = main_df_agri.get('region', 'Global')
    main_df_agri['country'] = main_df_agri.get('country', 'Global')
    main_df_agri[['yday_close_price_date', 'yday_close_price']] = (
        main_df_agri['ticker'].apply(_get_price).to_list()
    )

    main_df_agri.to_csv(DATA_INTERMEDIATE / 'summary_agri.csv', index=False)
    save_screened_etfs(main_df_agri, 'agri', portfolio_year=2026)
    print(f"Saved {len(main_df_agri)} agri ETCs")
    display(main_df_agri[['ticker', 'name', 'ter', 'dividends', 'size', 'last_year_return_per_risk',
                           'available_on_investengine', 'yday_close_price']])
else:
    print(f"Warning: {cmd_raw_path} not found — run notebook 01 first to scrape commodities data.")

############# COMBINED SUMMARY #############
summary_df = pd.concat(
    [df for df in [main_df_equities, main_df_bonds, main_df_pm, main_df_energy, main_df_agri] if not df.empty],
    ignore_index=True,
)
summary_df.to_csv(DATA_INTERMEDIATE / 'summary_all.csv', index=False)
print(f"Saved {len(summary_df)} total ETFs to summary_all.csv")
summary_df[['asset_class', 'region_category', 'intra_region_category', 'ticker', 'name',
            'ter', 'available_on_investengine']]